# Create Conjunction List
Find all conjunctions of in situ Swarm satellites with GNSS scintillation measurments for a particular reciever.

In [1]:
import numpy as np
import datetime as dt
import pymap3d as pm
import pandas as pd
import csv
import argparse
from satgroundconj.conjunction import SatConj
from gnssutils.map_prn import prn2norad
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

Define the CHAIN reciever and look up the coordinates in the site file.

In [2]:
receiver_name = 'res'

with open('CHAINsites.txt', 'r') as cf:
    chain_data = csv.reader(cf)
    for row in chain_data:
        if row[1] == receiver_name:
            receiver_site = [float(row[2]), float(row[3]), 0.]

print(receiver_site)

[74.746627, 264.997469, 0.0]


Define some basic parameters of the conjunction search.

In [3]:
output_file = f'insitu_gnss_conj_{receiver_name}.txt'

starttime = dt.datetime(2016,1,1, tzinfo=dt.timezone.utc)
endtime = dt.datetime(2016,1,2, tzinfo=dt.timezone.utc)

elev_cutoff = 30.    # Minimum elevation angle to consider for GNSS (deg)
tstep = 10.          # Time step for calculating satellite position (s)
conj_threshold = 10. # Angular separation threshold to count conjunctions (deg)


## In Situ Satellites
Define satellites, their NORAD numbers, and the period when they were orbiting.

In [4]:
# Swarm A, B, C; LLITED A, B
insitu_sats = [{'name': 'Swarm A', 'NORAD': 39452, 'start': dt.datetime(2013,12,2, tzinfo=dt.timezone.utc), 'end': dt.datetime.now(tz=dt.timezone.utc)},
               {'name': 'Swarm B', 'NORAD': 39451, 'start': dt.datetime(2013,12,2, tzinfo=dt.timezone.utc), 'end': dt.datetime.now(tz=dt.timezone.utc)},
               {'name': 'Swarm C', 'NORAD': 39453, 'start': dt.datetime(2013,12,2, tzinfo=dt.timezone.utc), 'end': dt.datetime.now(tz=dt.timezone.utc)}]

Find passes for in situ satellites.

In [5]:
# Generate lists of all passes for in situ satellites
print('Finding passes for In Situ...')
passes_insitu = list()
for sat in insitu_sats:
    print(sat['name'], 'NORAD ID:', sat['NORAD'])
    # Initialize conjunction object
    insitu = SatConj(receiver_site, sat['NORAD'], tolerance=90.-elev_cutoff, deltime=tstep, tledb='/Users/e30737/Desktop/Data/TLE/tle_full.db')
    # Find correct time bounds
    if starttime > sat['start']:
        st = starttime.replace(tzinfo=None)
    else:
        st = sat['start'].replace(tzinfo=None)
    if endtime < sat['end']:
        et = endtime.replace(tzinfo=None)
    else:
        et = sat['end'].replace(tzinfo=None)
    # Calculate conjunctions with ground site
    passes = insitu.conjunctions(st, et)
    # Add satellite name and ID to each pass item
    for p in passes:
        p['name'] = sat['name']
        p['NORAD'] = sat['NORAD']
    passes_insitu.extend(passes)

print(len(passes_insitu))

Finding passes for In Situ...
Swarm A NORAD ID: 39452
Swarm B NORAD ID: 39451
Swarm C NORAD ID: 39453
10


## GNSS Satellites

In [6]:
# Generate lists of all passes for GNSS satellites
print('Finding passes for GNSS...')
passes_gnss = list()
for gnum in range(1,32):
    prn = f'G{gnum:02d}'
    sat_id = prn2norad(prn, starttime)   # This only finds the NORAD number for a set date - will be a problem when searching across large time periods because these change
    print(prn, 'NORAD ID:', sat_id)
    if not sat_id:
        continue

    # Initalize conjunction object
    gnss = SatConj(receiver_site, sat_id, tolerance=90.-elev_cutoff, deltime=tstep, tledb='/Users/e30737/Desktop/Data/TLE/test8.db')
    # Find the correct time bounds
    if starttime > sat['start']:
        st = starttime.replace(tzinfo=None)
    else:
        st = sat['start'].replace(tzinfo=None)
    if endtime < sat['end']:
        et = endtime.replace(tzinfo=None)
    else:
        et = sat['end'].replace(tzinfo=None)
    # Calculate conjunctions with ground site
    passes = gnss.conjunctions(st, et)
    # Add satellite name and ID to each pass item
    for p in passes:
        p['PRN'] = prn
        p['NORAD'] = sat_id
    passes_gnss.extend(passes)

print(len(passes_gnss))

Finding passes for GNSS...
G01 NORAD ID: 37753
G02 NORAD ID: 28474
G03 NORAD ID: 40294
index out of bounds
G04 NORAD ID: None


/Users/e30737/Desktop/Software/gnssutils/src/gnssutils/map_prn.py:103: UserWarning: No PRN G04 available for 2016-01-01 00:00:00+00:00!
  warnings.warn(f'No PRN {prn} available for {date}!')


G05 NORAD ID: 35752
G06 NORAD ID: 39741
G07 NORAD ID: 32711
G08 NORAD ID: 40730
G09 NORAD ID: 40105
G10 NORAD ID: 41019
G11 NORAD ID: 25933
G12 NORAD ID: 29601
G13 NORAD ID: 24876
G14 NORAD ID: 26605
G15 NORAD ID: 32260
G16 NORAD ID: 27663
G17 NORAD ID: 28874
G18 NORAD ID: 26690
G19 NORAD ID: 28190
G20 NORAD ID: 26360
G21 NORAD ID: 27704
G22 NORAD ID: 28129
G23 NORAD ID: 28361
G24 NORAD ID: 38833
G25 NORAD ID: 36585
G26 NORAD ID: 40534
G27 NORAD ID: 39166
G28 NORAD ID: 26407
G29 NORAD ID: 32384
G30 NORAD ID: 39533
G31 NORAD ID: 29486
53


## Calculate Conjunctions

Haversine equation for calculating the angular distance between two sky points.

In [7]:
def haversine(phi1, phi2, lam1, lam2):
    
    p1 = np.deg2rad(phi1)
    p2 = np.deg2rad(phi2)
    l1 = np.deg2rad(lam1)
    l2 = np.deg2rad(lam2)
    
    dist = 2*np.arcsin(np.sqrt(np.sin((p2-p1)/2.)**2 + np.cos(p1)*np.cos(p2)*np.sin((l2-l1)/2.)**2))
    
    return np.rad2deg(dist)
 

Find conjunctions by iterating through every pair of in situ passes and gnss passes and selecting times when the following conditions are met:
1. The two passes occur at the same time (at least one time stamp the same between the two passes).
2. The closest the two satellites get during the pass is less than the threshold separation.

**Note** We might be able to do something clever where the initial selection of passes is at a relatively coarse resolution (1 min) but recalculate the position of the satellites at finer resolutions to better compute how closely they intersect.  That would probably involve calling the TLE propigation module somewhere in here?

In [8]:
print('Calculating conjunctions...')

conjlist = list()

# Cycle through all identified gnss and in situ passes
for pi in passes_insitu:
    for pg in passes_gnss:

        # Find common times between the two passes (the majority of pairs will have none)
        common_times = np.argwhere(pi['time'][:,None]==pg['time'][None,:])
        # If no common times, continue
        if len(common_times) == 0:
            continue

        # Pull out time and positions for both in situ and gnss passes at the common times
        ctimes = pi['time'][common_times[:,0]]
        xi, yi, zi = pi['position'][common_times[:,0],:].T
        xg, yg, zg = pg['position'][common_times[:,1],:].T

        # Calculate angular position
        ai, ei, ri = pm.ecef2aer(xi, yi, zi, *receiver_site)
        ag, eg, rg = pm.ecef2aer(xg, yg, zg, *receiver_site)

        # Calculate angular distance
        dist = haversine(ai, ag, ei, eg)
        mindist = np.min(dist)    # Minimum distance

        # If minimum distance less than threshold, add pair of passes to conjunction list
        if mindist < 10.:
            pass_info = pd.DataFrame({'time':ctimes, 'xi':xi, 'yi':yi, 'zi':zi, 'xg':xg, 'yg':yg, 'zg':zg, 'dist':dist})
            conj = {'Name':pi['name'], 'InSituNORAD':pi['NORAD'], 'PRN':pg['PRN'], 'GNSSNORAD':pg['NORAD'], 'mindist':mindist, 'pass_info':pass_info}
            conjlist.append(conj)

total_conjunctions = len(conjlist)
conjdf = pd.DataFrame(conjlist)

print('Total Conjunctions:', total_conjunctions)


Calculating conjunctions...
Total Conjunctions: 12


In [9]:
conjdf

,Name,InSituNORAD,PRN,GNSSNORAD,mindist,pass_info
0,Swarm A,39452,G03,40294,5.438949,time xi ...
1,Swarm A,39452,G17,28874,2.553756,time xi ...
2,Swarm A,39452,G12,29601,1.876033,time xi ...
3,Swarm A,39452,G25,36585,3.428291,time xi ...
4,Swarm A,39452,G31,29486,9.394508,time xi ...
5,Swarm B,39451,G26,40534,2.565639,time xi ...
6,Swarm B,39451,G07,32711,7.116806,time xi ...
7,Swarm B,39451,G30,39533,9.901981,time xi ...
8,Swarm B,39451,G02,28474,9.261373,time xi ...
9,Swarm B,39451,G09,40105,7.612494,time xi ...


In [10]:
conjdf['pass_info'].iloc[2]

,time,xi,yi,zi,xg,yg,zg,dist
0,2016-01-01 11:49:00,-383909.125663,-1.159162e+06,6.705995e+06,-9.974232e+06,-2.072691e+07,1.336118e+07,130.630129
1,2016-01-01 11:49:10,-388877.821375,-1.233995e+06,6.692371e+06,-9.973830e+06,-2.074395e+07,1.333542e+07,127.893040
2,2016-01-01 11:49:20,-393907.065241,-1.308667e+06,6.677906e+06,-9.973437e+06,-2.076097e+07,1.330963e+07,124.266193
3,2016-01-01 11:49:30,-398995.987762,-1.383169e+06,6.662603e+06,-9.973053e+06,-2.077795e+07,1.328382e+07,119.501170
4,2016-01-01 11:49:40,-404143.698243,-1.457491e+06,6.646464e+06,-9.972677e+06,-2.079490e+07,1.325797e+07,113.292828
5,2016-01-01 11:49:50,-409349.284941,-1.531623e+06,6.629491e+06,-9.972311e+06,-2.081182e+07,1.323210e+07,105.287905
6,2016-01-01 11:50:00,-414611.815213,-1.605557e+06,6.611685e+06,-9.971954e+06,-2.082871e+07,1.320620e+07,95.149927
7,2016-01-01 11:50:10,-419930.335662,-1.679283e+06,6.593049e+06,-9.971605e+06,-2.084556e+07,1.318027e+07,82.782930
8,2016-01-01 11:50:20,-425303.872298,-1.752792e+06,6.573585e+06,-9.971265e+06,-2.086238e+07,1.315431e+07,68.783276
9,2016-01-01 11:50:30,-430731.430697,-1.826075e+06,6.553296e+06,-9.970934e+06,-2.087917e+07,1.312832e+07,54.667079
